In [53]:
from qptifffile import QPTiffFile
import matplotlib.pyplot as plt
import numpy as np




In [54]:
f = QPTiffFile('/Users/chesterchan/Chester_phenocycler/data/neg ctr_Scan1.qptiff')

In [55]:
print(f.get_biomarkers())

['DAPI', 'Opal 690', 'Opal 620', 'Opal 780', 'Opal 520', 'Sample AF']


In [56]:
dapi_image = f.read_region('DAPI')
opal780_image = f.read_region('Opal 780')

In [ ]:
plt.imshow(dapi_image, cmap='gray')
plt.show()

In [60]:
print(opal780_image)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [61]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

opal780_image.shape


(84960, 36480)

In [62]:
H, W = opal780_image.shape

In [63]:
tile_size = 256
stride = 128
tiles = []

for x in range(0, W-tile_size+1, stride):
    for y in range(0, H-tile_size+1, stride):
        tile = opal780_image[y:y+tile_size, x:x+tile_size]
        tiles.append(tile)

tiles[0].shape

(256, 256)

In [64]:
def background_cleaner(tile, signal_threshold = 0.1, tile_threshold = 0.25):
    if tile.dim() == 3:
        tile = tile[0]
    else:
        pass

    average_signal = (tile > signal_threshold).float().mean()

    return average_signal > tile_threshold

In [65]:
filtered_tiles = []
for tile in tiles:
    tile = torch.from_numpy(tile)
    if background_cleaner(tile):   
        filtered_tiles.append(tile)


In [66]:
from torch.utils.data import DataLoader, Dataset
class DataCharacteristic(Dataset):
    def __init__(self, filteredtiles):
        self.data = filteredtiles

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self,idx):
        return self.data[idx]

processed_tiles = DataCharacteristic(filtered_tiles)

In [85]:
loader = DataLoader(processed_tiles, batch_size= 4, shuffle=True)
batch.shape

torch.Size([4, 256, 256])

In [86]:
dataiter = iter(processed_tiles)
batch = next(dataiter)

print(torch.min(batch), torch.max(batch))

tensor(0, dtype=torch.uint8) tensor(6, dtype=torch.uint8)


In [87]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(256*256, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 24),
            nn.ReLU(),
            nn.Linear(24, 3)
        )
        self.decoder = nn.Sequential(
            nn.Linear(3, 24),
            nn.ReLU(),
            nn.Linear(24, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 256*256),
        )
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [90]:
model = Autoencoder()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

In [92]:
num_epochs = 10
outputs = []
for epoch in range(num_epochs):
    for batch in loader:
        batch = batch.float()
        recon = model(batch)
        loss = criterion(recon, batch.view(batch.size(0), -1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch:{epoch+1}, Loss:{loss.item():.4f}')
    outputs.append((epoch,batch,recon))



KeyboardInterrupt: 